# Scoring Alignment Data

Given gold standard data and a hypothesized set of alignments, we can score the alignments. Additionally, we can break down the scores by book, score by content ("essential") terms only, and investigate specific verse alignments. 

This notebook scores BSB data aligned with `eflomal` against the gold standard data.

## Metrics

* Precision: what percent of predicted alignments are correct?
* Recall: what percent of correct alignments are predicted?
* F1: the harmonic mean of precision and recall, one useful summary measure.
* AER (Alignment Error Rate): 1 - Precision

## Instantiating a Scorer

In [2]:
from biblealignlib.burrito import CLEARROOT, AlignmentSet
from biblealignlib.autoalign import scorer
targetlang, targetid, sourceid = ("eng", "BSB", "SBLGNT")
alsetref = AlignmentSet(targetlanguage=targetlang,
        targetid=targetid,
        sourceid=sourceid,
        langdatapath=(CLEARROOT / f"alignments-{targetlang}/data"))

In [3]:
# this directory should already exist and have burrito format alignments
condition = "notebookcheck"
conditiondir = alsetref.langdatapath.parent / f"exp/{targetid}/{condition}"
print(f"Conditiondir: {conditiondir}")
sc = scorer.Scorer(referenceset=alsetref, 
                   hypothesispath=(conditiondir / f"{sourceid}-{targetid}-eflomal.json"),
                   hypothesisaltid="eflomal")

Conditiondir: /Users/sboisen/git/Clear-Bible/alignments-eng/exp/BSB/notebookcheck

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/BSB/nt_BSB.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.toml
        
----- Hypothesis data: <AlignmentSet: eng, SBLGNT-BSB-eflomal>

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/BSB/nt_BSB.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/exp/BSB/notebookcheck/SBLGNT-BSB-eflomal.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/SBLGNT-BSB-manual.toml
        
Dropping 14346 bad

## Diagnosing Verse Scores

You can score an individual verse as well as displaying the alignments for diagnostic purposes. 

In [4]:
# R = Reference, H = Hypothesis
bcvid = "41004002"
print(sc.score_verse(bcvid).summary())
sc.verse_dataframe(bcvid)

41004002: AER=0.1538	P=0.8462	R=0.7857	F1=0.8148


,And,He,taught,them,many,things,in,parables,and,in.1,His,teaching,He.1,said
καὶ,R-H,,,,,,,,,,,,,
ἐδίδασκεν,,R--,R-H,,,,,,,,,,,
αὐτοὺς,,,,R-H,,,,,,,,,,
ἐν,,,,,,,R-H,,,,,,,
παραβολαῖς,,,,,,,,R-H,,,,,,
πολλά,,,,,R-H,R-H,,,,,,,,
καὶ,,,,,,,,,R-H,,,,,
ἔλεγεν,,,,,,,,,,,,,R--,R-H
αὐτοῖς,,,,,,,,,,,,,,
ἐν,,,,,,,,,,R-H,,,,


In [ ]:
bcvid = "41004004"
print(sc.score_verse(bcvid).summary())
sc.verse_dataframe(bcvid)

## Scoring a Group of Verses

You can aggregate the scores for a set of verses defined by a common BCV prefix: for example, a chapter or a book. 

In [ ]:
gs = sc.score_group("41004")
gs = sc.score_group("41004")print(gs.summary())

In [ ]:
for vs in gs.verse_scores: print(vs.summary())

## Scoring the Whole Corpus

In [ ]:
print(sc.score_all().summary())

In [ ]:
# you can just score "essential" (content) tokens
print(sc.score_all(essential=True).summary())

You can log a condition and its score for later review: this is especially useful for baseline conditions you'll want to compare later.

The score is output as a row to `exp/$TARGETID/scorelog.tsv`. 

In [ ]:
sc.log_score(comment="optional comments on the condition")
# or essential scores
# sc.log_score(essential=True, comment="optional comments on the condition")

### Scoring Partial Data

If you've only aligned a partial corpus (see ...), you'll need to generate a partial gold standard file TBD. 

Given the partial gold standard data and aligner output, scoring should work as described above. 